# Μάθημα 11 - Πρωτόκολλο Πράκτορα προς Πράκτορα (A2A)


## Ρύθμιση


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Τι είναι το Πρωτόκολλο A2A;

Το **Πρωτόκολλο Agent-to-Agent (A2A)** είναι ένα ανοιχτό πρότυπο που επιτρέπει σε πράκτορες τεχνητής νοημοσύνης να επικοινωνούν,
να ανακαλύπτουν ο ένας τον άλλον και να συνεργάζονται — ακόμη και όταν έχουν υλοποιηθεί σε διαφορετικά πλαίσια ή φιλοξενούνται
από διαφορετικές υπηρεσίες.

Βασικές έννοιες:

- **Ανακάλυψη** – Οι πράκτορες δημοσιεύουν μια *Κάρτα Πράκτορα* που περιγράφει τις δυνατότητές τους, καθιστώντας
  εύκολο για άλλους πράκτορες (ή ορχηστρωτές) να βρουν τον κατάλληλο ειδικό για μια εργασία.
- **Ανταλλαγή Μηνυμάτων** – Οι πράκτορες ανταλλάσσουν δομημένα μηνύματα μέσω ενός κοινού πρωτοκόλλου, ώστε ένα
  αίτημα από έναν πράκτορα να μπορεί να γίνει κατανοητό και να εκπληρωθεί από έναν άλλο ανεξάρτητα από την εσωτερική
  υλοποίησή του.
- **Κύκλος Ζωής Εργασίας** – Το A2A ορίζει καταστάσεις όπως *υποβλήθηκε*, *σε εξέλιξη*, *ολοκληρώθηκε* και
  *απέτυχε*, παρέχοντας στον ορχηστρωτή πλήρη ορατότητα στο πώς εξελίσσεται μια ανατεθειμένη εργασία.

Σε αυτό το μάθημα προσομοιώνουμε τη συνεργασία με στυλ A2A συνδέοντας τρεις εξειδικευμένους ταξιδιωτικούς πράκτορες
σε μια ροή εργασίας όπου κάθε πράκτορας συνεισφέρει την εξειδίκευσή του και μεταβιβάζει τα αποτελέσματα στον επόμενο.


## Δημιουργία Εξειδικευμένων Ταξιδιωτικών Πρακτόρων


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Συνεργασία Πολλών Πρακτόρων μέσω Ροής Εργασίας

Συνδέουμε τους τρεις πράκτορες σε μια διαδοχική ροή εργασίας που αντικατοπτρίζει τη διαβίβαση μηνυμάτων A2A:

1. **CurrencyExchangeAgent** λαμβάνει το αίτημα του χρήστη και παρέχει οδηγίες σχετικά με το συνάλλαγμα.
2. **ActivityPlannerAgent** λαμβάνει το εμπλουτισμένο πλαίσιο και προσθέτει προτάσεις δραστηριοτήτων.
3. **TravelManagerAgent** συνθέτει και τις δύο εισροές σε ένα τελικό ταξιδιωτικό ενημερωτικό σημείωμα.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Κατανόηση του A2A σε περιβάλλον παραγωγής

Σε περιβάλλον παραγωγής, το πρωτόκολλο A2A ξεκλειδώνει ισχυρά σενάρια μεταξύ υπηρεσιών:

| Δυνατότητα | Περιγραφή |
|---|---|
| **Διαλειτουργικότητα μεταξύ πλαισίων** | Ένας πράκτορας που έχει κατασκευαστεί με ένα πλαίσιο μπορεί να αναθέσει εργασίες σε έναν πράκτορα κατασκευασμένο με οποιοδήποτε άλλο πλαίσιο συμβατό με A2A, επιτρέποντας την πραγματική διαλειτουργικότητα μεταξύ οργανισμών. |
| **Όρια υπηρεσιών** | Οι πράκτορες μπορούν να βρίσκονται σε ξεχωριστές μικροϋπηρεσίες, περιοχές cloud ή ακόμη και διαφορετικούς οργανισμούς, ενώ εξακολουθούν να συνεργάζονται απρόσκοπτα. |
| **Δυναμική ανακάλυψη** | Ένας ορχηστρωτής μπορεί να ερωτήσει ένα μητρώο Agent Card κατά το χρόνο εκτέλεσης για να βρει τον πλέον κατάλληλο ειδικό για μια δεδομένη υποεργασία. |
| **Ροή & ειδοποιήσεις push** | Το A2A υποστηρίζει τα Server-Sent Events (SSE) για ενημερώσεις προόδου σε πραγματικό χρόνο και ειδοποιήσεις push για εργασίες μεγάλης διάρκειας. |

Η ροή εργασίας που δημιουργήσαμε παραπάνω είναι μια απλουστευμένη, εντός διεργασίας έκδοση αυτού του προτύπου. Σε μια πραγματική
ανάπτυξη, κάθε πράκτορας θα εκθέσει ένα HTTP endpoint, θα δημοσιεύσει ένα Agent Card, και θα επικοινωνούσε
μέσω του πρωτοκόλλου A2A JSON-RPC.


## Περίληψη

Σε αυτό το μάθημα μάθατε:

1. **Τι είναι το πρωτόκολλο A2A** — ένα ανοιχτό πρότυπο για την ανακάλυψη μεταξύ πρακτόρων, την ανταλλαγή μηνυμάτων και τη διαχείριση εργασιών.
2. **Πώς να δημιουργήσετε εξειδικευμένους πράκτορες** — ένας πράκτορας ανταλλαγής νομισμάτων, ένας πράκτορας σχεδιασμού δραστηριοτήτων και ένας ορχηστρωτής Διαχείρισης Ταξιδιών.
3. **Πώς να συνδέσετε πράκτορες σε μια ροή εργασίας** — χρησιμοποιώντας `WorkflowBuilder` για τη μοντελοποίηση της διαδοχικής διαβίβασης μηνυμάτων μεταξύ πρακτόρων.
4. **Πώς λειτουργεί το A2A σε παραγωγή** — επιτρέποντας συνεργασία μεταξύ διαφορετικών πλαισίων και υπηρεσιών με δυναμική ανακάλυψη και ροή ενημερώσεων.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Παρά τις προσπάθειές μας για ακρίβεια, παρακαλούμε να σημειώσετε ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν σφάλματα ή ανακρίβειες. Το πρωτότυπο έγγραφο στη γλώσσα του πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες συνιστάται επαγγελματική μετάφραση από ανθρώπινο μεταφραστή. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
